# Data Splitting and Cross Validation

This notebook will demonstrate dataset preparation and splitting strategies for performing validation methods such as holdout or cross-validation. We will work with the Telco Churn dataset. 

We will then examine different validation strategies: a holdout split and cross-validation. Our aim is to understand why multiple, well-constructed validation splits provide a more trustworthy picture of model performance, and how preserving class balance helps ensure fair evaluation when classes are imbalanced.

The data set includes information about:

- Customers who left within the last month – the column is called `Churn`
- Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies
- Customer account information – how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges
- Demographic info about customers – gender, age range, and if they have partners and dependents


### Loading Dataset and Defining the Task

The aim is to predict customer churn (binary classification). We first load the dataset and check the target distribution to understand class balance.

In [1]:
# install the libraries to be used:
import pandas as pd, numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer   # pipelining modules

In [2]:
df = pd.read_csv("./datasets/telco_churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info(), df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


(None,
 Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
        'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
        'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
       dtype='object'))

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


Let's also fix the column data types

In [5]:
# numeric columns
df['MonthlyCharges'] = df['MonthlyCharges'].astype('float')

# if you observe this column, you will find it contains some blank strings ' '
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')     # this will convert ' ' to NaN
df['TotalCharges'].isna().sum()

np.int64(11)

In [6]:
# let's drop the nan values now
df = df.dropna(subset=['TotalCharges'])
df['TotalCharges'].isna().sum()

np.int64(0)

In [7]:
# categorical columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('customerID')
df[cat_cols] = df[cat_cols].astype('category')

In [8]:
# the TotalCharges columns contains some blank strings ' '
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerID        7032 non-null   object  
 1   gender            7032 non-null   category
 2   SeniorCitizen     7032 non-null   int64   
 3   Partner           7032 non-null   category
 4   Dependents        7032 non-null   category
 5   tenure            7032 non-null   int64   
 6   PhoneService      7032 non-null   category
 7   MultipleLines     7032 non-null   category
 8   InternetService   7032 non-null   category
 9   OnlineSecurity    7032 non-null   category
 10  OnlineBackup      7032 non-null   category
 11  DeviceProtection  7032 non-null   category
 12  TechSupport       7032 non-null   category
 13  StreamingTV       7032 non-null   category
 14  StreamingMovies   7032 non-null   category
 15  Contract          7032 non-null   category
 16  PaperlessBilling  7032 non-nu

In [9]:
# df.to_parquet('./datasets/telco_churn_cleaned.parquet')      # save cleaned DF as parquet to retain the preprocessed info such as dtypes

#### Defining Features and Target

In [10]:
y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn", "customerID"])
X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


### Basic Transformation and Splitting Pipeline

Since our dataset has both numerical and categorical features, we will use a `ColumnTransformer` to apply scaling and encoding only where needed

This is wrapped inside a `Pipeline`, which ensures that all preprocessing steps are correctly applied inside each validation split, preventing leakage. While we could manually scale features before splitting, this would risk exposing information from the validation set to the model. 

By using a pipeline, we keep preprocessing tidy, reproducible, and evaluation-safe, then move on to comparing different validation strategies

In [11]:
numeric_features = X.select_dtypes(include=["int", "float"]).columns
categorical_features = X.select_dtypes(include=["category"]).columns

# ColumnTransformer allows us to apply different preprocessing to different column types
preprocess = ColumnTransformer(
    transformers=[("num", StandardScaler(), numeric_features),("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)]
)

# Pipeline joins preprocessing and modelling into a single estimator
model = Pipeline([("preprocess", preprocess), ("clf", LogisticRegression(max_iter=1000))])

The pipeline ensures that every fold in cross-validation receives its own independent preprocessing:
- When a fold is designated as training, the scaler and encoder are fitted only on that fold
- When the model predicts on the validation fold, transformation is applied using only what was learned from its training fold

### Using Validation Splits

#### Holdout

We split the data before preprocessing to avoid leaking patterns and stratify to preserve class proportions in both sets

In [12]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=1)

y_tr.value_counts(normalize=True), y_te.value_counts(normalize=True)    # class distribution

(Churn
 0    0.7328
 1    0.2672
 Name: proportion, dtype: float64,
 Churn
 0    0.739872
 1    0.260128
 Name: proportion, dtype: float64)

In [13]:
model.fit(X_tr, y_tr)

# predictions and probabilities on the hold-out set
preds = model.predict(X_te)
preds_proba = model.predict_proba(X_te)[:, 1]

# evaluation metrics
print("Accuracy :", accuracy_score(y_te, preds))
print("Recall   :", recall_score(y_te, preds))
print("ROC AUC  :", roc_auc_score(y_te, preds_proba))

Accuracy : 0.7960199004975125
Recall   : 0.5327868852459017
ROC AUC  : 0.8472346892174925


Accuracy comes at around 80% but the recall is not that great. Recall is an important metric in churn prediction (why?). What will happen if we change the `random_state` during `train_test_split`? Try a few varying seed values. Obviously, due to the very high class imbalance, the recall is modest despite a decent accuracy. We can probably handle this by using some resampling techniques (as this is not the objective here, the learner is encouraged to try this as an exercise).

Note that we are treating the test set as a validation set here

Let's try cross-validation next

#### Cross-Validation

Let's try KFold CV and Stratified KFold CV to see if the final distributions differ

In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)    
# shuffle argument shuffles the data before split to ensure the sample gets mixed up
# this is good if the distribution of classes in the data is not uniform across the rows and some classes are more concentrated in a small group of rows


# in CV, we are using only the training set to create validation folds, and the test set is kept separate for final evaluation
for fold, (_, test_idx) in enumerate(kf.split(X_tr)):
    print(f"Fold {fold + 1}")
    print(y.iloc[test_idx].value_counts(normalize=True))

Fold 1
Churn
0    0.738667
1    0.261333
Name: proportion, dtype: float64
Fold 2
Churn
0    0.736
1    0.264
Name: proportion, dtype: float64
Fold 3
Churn
0    0.744889
1    0.255111
Name: proportion, dtype: float64
Fold 4
Churn
0    0.731556
1    0.268444
Name: proportion, dtype: float64
Fold 5
Churn
0    0.726222
1    0.273778
Name: proportion, dtype: float64


In [15]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

for fold, (_, test_idx) in enumerate(skf.split(X_tr, y_tr)):
    print(f"Fold {fold + 1}")
    print(y.iloc[test_idx].value_counts(normalize=True))

Fold 1
Churn
0    0.751111
1    0.248889
Name: proportion, dtype: float64
Fold 2
Churn
0    0.744889
1    0.255111
Name: proportion, dtype: float64
Fold 3
Churn
0    0.739556
1    0.260444
Name: proportion, dtype: float64
Fold 4
Churn
0    0.698667
1    0.301333
Name: proportion, dtype: float64
Fold 5
Churn
0    0.743111
1    0.256889
Name: proportion, dtype: float64


The distribution is a little bit better with stratified splitting. If the dataset was more segregated, we would have preferred the stratified split. However, here we can simply choose KFold.

In [16]:
# cross_val_score calls fit() and predict() inside each fold.
# the pipeline ensures preprocessing happens correctly within each fold's boundaries.
scores = cross_val_score(model, X_tr, y_tr, cv=kf, scoring="recall")    # since we are trying to maximise recall
scores

array([0.56776557, 0.57192982, 0.5751634 , 0.53636364, 0.53074434])

An important point to note here is that there are two ways we can use cross-validation

- First, to decide whether to choose a model. We apply cross-validation folds only on the training portion of the data. Once we know the model is suited for the task, we train the actual model on the whole training data normally. This is preferable when we have a sufficient amount of data.

- Second, to train the final model itself using cross-validation. Here, we use the `return_estimator` argument in the `cross_validate()` function, which returns the estimators as well from each fold. We can directly use the best estimator from these and perform a final evaluation using the test set. However, these estimators are each trained on only a fraction of the training data, and not guaranteed to be actually the best model. Although not recommended, this might prove useful in cases where we have access to only a small amount of data.

As we will see, this is not the case with `GridSearchCV`, which automatically refits the best model on the full training set, so the returned best_estimator_ is already ready for final testing.

In [17]:
cv_results = cross_validate(model, X_tr, y_tr, cv=kf, scoring="accuracy", return_estimator=True)  # this gives models from each fold

In [18]:
best_idx = scores.argmax()
best_cv_model = cv_results["estimator"][best_idx]
best_cv_model

,steps,"[('preprocess', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


Here, you can see the entire pipeline - preprocessing, scaling (separate for numeric and categorical features) and the final model

In [19]:
# let's try predicting using the final model
y_pred = best_cv_model.predict(X_te)

# evaluation
print("Accuracy :", accuracy_score(y_te, y_pred))
print("Recall   :", recall_score(y_te, y_pred))

Accuracy : 0.7924662402274343
Recall   : 0.5218579234972678


To summarise, we applied preprocessing using a pipeline to avoid leakage, and used different validation techniques for logistic regression. K-fold CV is a good validation method, but in case of imbalanced and skewed data, stratified K-fold CV may also prove useful.

Note that the performance of our final model remained comparatively similar before and after using cross-validation. This shows that cross-validation is not some magical technique that will improve your model by a lot. It just helps our model generalise better and serve as an early warning for overfitting or underfitting.